PRESS describes the decisions and actions of the ball carrier through separately fitted components: the probability of passing or carrying, the risk of losing the ball in a one-second carry, the destination of a pass and its probability of completion. This page composes them into a single quantity, h, that answers one question: given the current state, how likely is the team to lose the ball during the carrier's next action? Because h is a probability, it has an absolute scale, which we divide into four bands. The bands are used in the [pressure case studies and player report](../07_pressure_case_studies/07_pressure_case_studies.ipynb).

## Definition

The quantity h measures the **fragility of team possession** at the state of the current ball carrier, namely the probability that the team loses the ball during the current action. Team possession is discretized into canonical actions

$$\mathcal A_0=\{\mathrm{carry}_{1s},\ \mathrm{pass}(z)\},$$

where a carry lasts one second and a pass runs until its completion. The one-second carry unit is inspired by action-based EPV models [@fernandez2019; @fernandez2021; @stalpaert2022]. Shots are outside the action set. At a canonical decision state $s$ of the team's sequence,

$$h(s)=P(\mathrm{possession\ loss\ during\ the\ current\ action}\mid s,\ \mathrm{team\ possession\ still\ active}).$$

This is a **discrete loss hazard on the team's action sequence**, indexed by decisions and not by frames or seconds. A successful carry continues control with the same player. A successful pass continues team possession with the receiver, so it is not a terminal event for h, although it ends the passer's individual possession. This is what distinguishes h from the individual terminal target of xLoss, introduced at the end of this page.

h is a **probability per decision**. It is not a rate per second or a forecast of loss by the end of the whole possession, and changing the carry horizon changes its scale. The score can be evaluated at any reliable tracking frame, where it averages over a learned pass/carry policy. It links spatial pressure to possession fragility through explicit choice and execution components: local geometry and kinematics enter execution, and passing options and context enter choice. The components describe an average player in the observed context, and a high value indicates difficult modeled alternatives under that policy and not inevitable failure. Loss labels and the conventions for clearances and out-of-play endings are documented in the [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd).


## Mathematical definition

Condition all quantities on the current state $s$. Four separately fitted PRESS components enter:

* $\pi(s)=P(A=\mathrm{pass}\mid s)$ is the probability of passing from the [action-selection model](../04_action_selection/04_action_selection.ipynb), and $1-\pi(s)$ the probability of carrying;
* $d(s)=P(\mathrm{loss\ within\ 1s}\mid A=\mathrm{carry},s)$ is the loss probability of the one-second carry, from the [carry-risk model](../01_ball_drive_risk/01_ball_drive_risk.ipynb);
* $q_j(s)=P(J=j\mid A=\mathrm{pass},s)$ is the probability of destination $j$, from the [pass-destination model](../03_transition_model/03_transition_model.ipynb);
* $p_j(s)=P(\mathrm{completion}\mid J=j,A=\mathrm{pass},s)$ is the completion probability of a pass to $j$, from the [pass-probability model](../02_pass_probability/02_pass_probability.ipynb).

Averaging pass failure over the destinations gives

$$f(s)=\sum_j q_j(s)\,\big(1-p_j(s)\big)=1-\sum_j q_j(s)\,p_j(s),$$

and the law of total probability over the two action branches gives

$$\boxed{h(s)=\pi(s)\,f(s)+\big(1-\pi(s)\big)\,d(s).}$$

The weighted pass contribution is $\pi f$ and the weighted carry contribution is $(1-\pi)d$; they add exactly to h and differ from the branch risks f and d themselves. A state can therefore have a high pass risk and a low h when passing is unlikely, and vice versa. All components are evaluated from the state at that frame, and the observed next choice and its outcome do not enter the formula. The sum $\sum_j q_jp_j$ is the expected pass success of the action-selection features, called `rppcf` in the code. Tracking velocities use the centered seven-frame smoother (±0.3 s at 10 Hz), so h is computed offline and is not a causal real-time estimator.

The composition is a project derivation, and the terminology follows the conditional failure probability of discrete-time survival analysis [@singer2003; @suresh2022discrete]. The internal physical tackle hazard is a rate and can exceed one, while h is a probability in $[0,1]$.


## Absolute bands

To give h an interpretable scale, consider repeated team actions with the same conditional loss probability h. The probability of avoiding loss over $n$ of them is $(1-h)^n$, and the **equivalent constant-risk half-life** is the number of actions that halves it:

$$n_{1/2}=\frac{\log(1/2)}{\log(1-h)},\qquad b_n=1-2^{-1/n},$$

where $b_n$ is the value of h whose half-life is exactly $n$ actions. The bands use the cutoffs $b_8$, $b_4$ and $b_2$. This is an interpretable project design choice, and the thresholds are not learned from outcomes. The API keeps the labels below.

| Band | h interval | Constant-risk half-life |
|---|---|---|
| None | $0\le h<0.082996$ | More than 8 decisions; infinite at h=0 |
| Low | $0.082996\le h<0.159104$ | $4<n_{1/2}\le8$ decisions |
| Medium | $0.159104\le h<0.292893$ | $2<n_{1/2}\le4$ decisions |
| High | $0.292893\le h\le1$ | At most 2 decisions |

The code uses the exact cutoffs, and a value on a cutoff enters the higher band. The cutoff for $n=1$, h=0.5, lies within High and does not define a fifth band. **None denotes the lowest modeled risk band and not the absence of defensive pressure.** The half-life is a scale in decisions and not in seconds, and it does not forecast how long the team keeps possession, since future states and their risks change.

## Numerical example

Suppose $\pi=0.4$, $d=0.1$, the destination probabilities $q_j$ are $(0.25,0.5,0.25)$ and the completion probabilities $p_j$ are $(0.7,0.4,0.2)$. The expected pass failure is $f=0.575$, so $h=0.6\times0.1+0.4\times0.575=0.29$, which lies in Medium. The inputs are synthetic.


In [ ]:
import numpy as np
from PRESS.utils.hazard import compute_hazard_raw
from PRESS.utils.possession import hazard_band_thresholds, hazard_to_band
from scripts.build_h_vignette_figures import main as rebuild_h_figures

transition = np.array([0.25, 0.50, 0.25])
pass_success = np.array([0.70, 0.40, 0.20])
h, drive_loss, pass_failure, weighted_pass_success = compute_hazard_raw(
    0.1, transition, pass_success, 0.4)
assert np.isclose(h, 0.29) and hazard_to_band(h) == "Medium"
print({"h": h, "carry contribution": 0.6 * drive_loss,
       "pass contribution": 0.4 * pass_failure, "band": hazard_to_band(h)})
print("Exact cutoffs:", hazard_band_thresholds())
rebuild_h_figures()


![Synthetic branch mixture and exact band boundaries.](../../figures/vignette_05_hazard_bands.png)


## Validation of the bands

The table pools **15,085 canonical held-out action states** from the five outer test pairs. The thresholds are fixed by the half-life construction and are not quantiles or boundaries selected on the outcomes. The observed target is the **failure of the current canonical action** and not the terminal individual loss.

| Band | Actions | Mean predicted h | Observed failures |
|---|---:|---:|---:|
| None | 8,716 | 2.94% | 3.37% |
| Low | 2,984 | 11.88% | 12.06% |
| Medium | 2,534 | 21.39% | 21.23% |
| High | 851 | 36.51% | 32.08% |

Observed failure increases monotonically across bands and falls inside the interval of each band. Predictions track the observed rates closely in None, Low and Medium, and High is somewhat pessimistic (36.51% predicted versus 32.08% observed).

![Observed action-level failure and mean predicted h for the fixed bands.](../../figures/h_band_validation.png)

Error bars resample **whole matches** 2,000 times with a fixed seed, keeping the cutoffs unchanged [@efron1993bootstrap]. Longer possessions supply several canonical actions, so the counts are not independent possessions. The [composed-risk benchmark](../../benchmarks/06_composed_risk.qmd) reports CE, gain over null and the pooled ten-quantile calibration curve, and no post-hoc calibrator is fitted on h.

Because h marginalizes over the choice, splitting the data by the **observed** pass or carry changes the conditioning, and h need not equal the event rate within each subset.


In [ ]:
from pathlib import Path
import pandas as pd
from scripts.build_pressure_report_figures import main as rebuild_pressure_figures

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
validation = pd.read_csv(root / "data/examples/pressure_report/band_validation.csv", keep_default_na=False)
assert validation["n"].sum() == 15085 and validation["losses"].sum() == 1465
assert validation["observed_loss"].is_monotonic_increasing
print(validation[["band", "n", "mean_h", "observed_loss"]].to_string(index=False))
rebuild_pressure_figures()


## Canonical decisions and extension to xLoss

The formula can be queried at any reliable tracking frame, while the bands and the benchmark are defined on canonical decisions, namely one-second carries and pass releases. Their segmentation is described in [Possessions and checkpoints](../setup/possession_and_checkpoint_setup.ipynb). An additional snapshot, for example in a short final stretch of a possession, uses the same canonical forecasts and does not create a training row or an independent exposure unit.

[Residual risk](../05_residual_risk/05_residual_risk.ipynb) completes the model toward the individual terminal loss,

$$\mathrm{xLoss}(s)=h(s)+\big(1-\pi(s)\big)\big(1-d(s)\big)\,c(s),$$

where $c(s)$ is the residual risk, the probability that the individual possession ends in a loss after a successful carry. The target is different from that of h: a completed pass ends the individual sequence without loss, while the team sequence underlying h continues. The h profile and its bands do not depend on this continuation model.
